Per-system identifier formats:

auth → firstname.lastname@companyname.com

file_access → U + zero-padded number (e.g. U00123)

privileged_command → adm-firstname.lastname (privileged humans) or svc-<function>-<3digit> (service_accounts/agents)

network_access → WKS-<4digit> (humans) or HOST-<function>-<3digit> (service_accounts/agents)

In [0]:
pip install faker

In [0]:
from faker import Faker

In [0]:
# Company domain
email_domain = "erspgroup.com"

# Global Counters
counters = {
    "u_counter": 0,
    "a_counter": 0,
    "m_counter": 0
}

fake_person = Faker()

In [0]:
def generate_email():
    return fake_person.first_name().lower() + "." + fake_person.last_name().lower() + "@" + email_domain

In [0]:
def generate_userid():
    counters["u_counter"] += 1
    return "U00" + str(counters["u_counter"])

In [0]:
def generate_access_name(entity_type, entity_role):
    counters["a_counter"] += 1
    if entity_type == "human":
        return "adm-" + str(counters["a_counter"])
    if entity_type == "service_account": 
        return "svc-" + entity_role + "-" + str(counters["a_counter"])
    if entity_type == "agent": 
        return "agt-" + entity_role + "-" + str(counters["a_counter"])

In [0]:
def generate_machine_name(entity_type, entity_role):
    counters["m_counter"] += 1
    if entity_type == "human":
        return "WKS-" + str(counters["m_counter"])
    if entity_type in ("service_account", "agent"): 
        return "HOST-" + entity_role + "-" + str(counters["m_counter"])

In [0]:
df = spark.read.table("entity_risk_platform.seed_data.entities")
all_entities = df.select("entity_id", "role", "entity_type", "tier")

auth: humans only, always

file_access: all entities, always

network_access: all entities, always

privileged_command:
Human: tier == "Manager" (any role) OR (role == "IT" AND tier == "Senior")
Non-human: role in ["ci-cd", "security-scanning", "backup-automation"]

In [0]:
new_system_identifier_list = []

In [0]:
def generate_identifier(entity_id, system_name, identifier): 
    return {"entity_id": entity_id, "system_name": system_name, "system_identifier": identifier}

def process_entity(entity):
    # Auth Identifier
    if(entity["entity_type"] == "human"):
        new_system_identifier_list.append(generate_identifier(entity["entity_id"], "auth", generate_email()))
    # File Identifier
    new_system_identifier_list.append(generate_identifier(entity["entity_id"], "file_access", generate_userid()))
    # Command Identifier
    if(entity["entity_type"] == "human"):
        if(entity["tier"] == "Manager" or (entity["tier"] == "Senior" and entity["role"] == "IT")):
            new_system_identifier_list.append(generate_identifier(entity["entity_id"], "privileged_command", generate_access_name(entity["entity_type"], entity["role"])))
    if(entity["entity_type"] in ("service_account", "agent")):
        if(entity["role"] in ["ci-cd", "security-scanning", "backup-automation"]):
            new_system_identifier_list.append(generate_identifier(entity["entity_id"], "privileged_command", generate_access_name(entity["entity_type"], entity["role"])))
    # Network Identifier
    new_system_identifier_list.append(generate_identifier(entity["entity_id"], "network_access", generate_machine_name(entity["entity_type"], entity["role"])))


In [0]:
for row in all_entities.collect():
    process_entity(row.asDict())
df = spark.createDataFrame(new_system_identifier_list)
df.show()

In [0]:
# Re-read the saved table to verify what's actually persisted
verify_df = spark.read.table("entity_risk_platform.seed_data.entity_system_identifiers")

# 1. Total row count
print("Total rows:", verify_df.count())

# 2. Row count per system_name — sanity check against expected counts
verify_df.groupBy("system_name").count().show()

# 3. Duplicate check — this is the important one.
#    Group by (system_name, system_identifier) and find any group with count > 1.
#    A clean result here should show ZERO rows.
from pyspark.sql import functions as F

duplicates = (
    verify_df.groupBy("system_name", "system_identifier")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate identifiers found:", duplicates.count())
duplicates.show(truncate=False)

# 4. Every entity_id should trace back to a real entity in `entities` — orphan check
entities_df = spark.read.table("entity_risk_platform.seed_data.entities")
orphans = verify_df.join(entities_df, on="entity_id", how="left_anti")
print("Orphaned entity_id references:", orphans.count())

# 5. Every human should have exactly one 'auth' row — spot check the rule you locked
human_ids = entities_df.filter(F.col("entity_type") == "human").select("entity_id")
auth_rows = verify_df.filter(F.col("system_name") == "auth")
humans_missing_auth = human_ids.join(auth_rows, on="entity_id", how="left_anti")
print("Humans missing an auth identifier:", humans_missing_auth.count())

# 6. No non-human should ever have an 'auth' row — the inverse check
non_human_auth = (
    verify_df.filter(F.col("system_name") == "auth")
    .join(entities_df, on="entity_id")
    .filter(F.col("entity_type") != "human")
)
print("Non-humans incorrectly given an auth identifier:", non_human_auth.count())